In [ ]:
from neo4j import GraphDatabase

# Function to connect to Neo4j
def connect_to_neo4j(uri, user, password):
    driver = GraphDatabase.driver(uri, auth=(user, password))
    return driver


# Connect to a Neo4j instance which enables Neo4j GDS, e. g. a local database
# adjust credentials according to your Neo4j instance
uri = "bolt://localhost:7689" 
user = "neo4j"
password = "password"
NEO4J_DB = "neo4j"

driver = connect_to_neo4j(uri, user, password)


In [20]:
# getting started with Neo4j Graph Data Science

from graphdatascience import GraphDataScience
gds = GraphDataScience(uri, auth=(user, password), database=NEO4J_DB)

# Check the installed GDS version on the server

print(gds.version())
assert gds.version() 

2.6.8


In [36]:
gds.run_cypher("""MATCH (n:Biological_sample)
               WHERE n.subjectid STARTS WITH "10" OR n.subjectid STARTS WITH "40" OR n.subjectid STARTS WITH "41"
               MATCH (m:Disease)
               OPTIONAL MATCH (n)-[r:HAS_DISEASE]->(m) 
               RETURN count(DISTINCT n) + count(DISTINCT m) as nodes, count(r) as relationships""")

,nodes,relationships
0,10859,94


In [ ]:
gds.run_cypher(""" MATCH (m:Disease) 
               RETURN count(m) as nodes
               //, count(r) as relationships""")

,nodes
0,10791


In [ ]:
## project training graph
## here, Biological_sample for training and test graph were not randomly sampled, but selected based on subjectid
## subjectid's may not be up-to-date anymore in newer dump-files for local databases
## -> please adjust the query accordingly or use the sampling approach in GDS_link_prediction_composite.py

G_train_exists = gds.run_cypher("""CALL gds.graph.exists("train_graph") YIELD exists""")

#pipe_cli_exists = gds.run_cypher("""CALL gds.pipeline.exists('pipe_cli') YIELD exists""")

if G_train_exists.iloc[0,0]==True:
    gds.graph.drop("train_graph")


#create graph projection
G_train, result = gds.graph.cypher.project("""
    MATCH (source)
    WHERE (source:Biological_sample AND source.subjectid STARTS WITH "10") OR
          (source:Biological_sample AND source.subjectid STARTS WITH "40") OR
          (source:Biological_sample AND source.subjectid STARTS WITH "41") OR
           source:Phenotype OR 
           source:Protein OR 
           source:Disease    
    OPTIONAL MATCH (source)-[r:HAS_PHENOTYPE|HAS_DAMAGE|HAS_PROTEIN|COMPILED_INTERACTS_WITH|HAS_DISEASE|IS_BIOMARKER_OF_DISEASE|HAS_PARENT]->(target)
    WHERE target:Phenotype OR                                                                      
            target:Gene OR
            target:Protein OR
            target:Disease                                                                                                                                                                                            
    RETURN gds.graph.project(
    'train_graph',
    source,
    target,
    {
    sourceNodeLabels: labels(source),
    //sourceNodeProperties: source { .subjectid, .id},                                           
    targetNodeLabels: labels(target),
    //targetNodeProperties: target { .id},                                           
    relationshipType: type(r),
    relationshipProperties: r { score: coalesce(r.score, 0.0) }
    },
    { undirectedRelationshipTypes: ['HAS_DISEASE']}                                    
    )""")

assert G_train.node_count() == result["nodeCount"]

NameError: name 'gds' is not defined

In [5]:
## project test graph

G_test_exists = gds.run_cypher("""CALL gds.graph.exists("test_graph") YIELD exists""")

#pipe_cli_exists = gds.run_cypher("""CALL gds.pipeline.exists('pipe_cli') YIELD exists""")

if G_test_exists.iloc[0,0]==True:
    gds.graph.drop("test_graph")

#create graph projection
G_test, result_test = gds.graph.cypher.project("""MATCH (source)
    WHERE (source:Biological_sample AND source.subjectid STARTS WITH "42") OR
          (source:Biological_sample AND source.subjectid STARTS WITH "43") OR
          (source:Biological_sample AND source.subjectid STARTS WITH "44") OR
           source:Phenotype OR
           source:Protein OR
           source:Disease                                                                                                                                  
    OPTIONAL MATCH (source)-[r:HAS_PHENOTYPE|HAS_DAMAGE|HAS_PROTEIN|COMPILED_INTERACTS_WITH|HAS_DISEASE|IS_BIOMARKER_OF_DISEASE|HAS_PARENT]->(target)
            WHERE target:Gene OR target:Protein OR target:Phenotype OR
            (source:Biological_sample AND target:Disease AND target.id STARTS WITH "DOID:4")
    RETURN gds.graph.project(
    'test_graph',
    source,
    target,
    {
    sourceNodeLabels: labels(source),
    targetNodeLabels: labels(target),
    relationshipType: type(r),
    relationshipProperties: r { score: coalesce(r.score, 0.0) }
    },
    { undirectedRelationshipTypes: ['HAS_DISEASE']}                                    
    )""")
    
assert G_test.node_count() == result_test["nodeCount"]

In [ ]:
## project control graph to facilitate later comparison between predicted and true links

if gds.run_cypher("""CALL gds.graph.exists("control_graph") YIELD exists""").iloc[0,0]==True:
    gds.graph.drop("control_graph")

#create graph projection
G_control, result_test = gds.graph.cypher.project("""MATCH (source)
    WHERE (source:Biological_sample AND source.subjectid STARTS WITH "42") OR
          (source:Biological_sample AND source.subjectid STARTS WITH "43") OR
          (source:Biological_sample AND source.subjectid STARTS WITH "44") OR
           source:Phenotype OR
           source:Protein OR
           source:Disease                                                                                                                                  
    OPTIONAL MATCH (source)-[r:HAS_PHENOTYPE|HAS_DAMAGE|HAS_PROTEIN|COMPILED_INTERACTS_WITH|HAS_DISEASE|IS_BIOMARKER_OF_DISEASE]->(target)
            WHERE target:Gene OR target:Protein OR target:Phenotype OR
            target:Disease
    RETURN gds.graph.project(
    'control_graph',
    source,
    target,
    {
    sourceNodeLabels: labels(source),
    targetNodeLabels: labels(target),
    relationshipType: type(r),
    relationshipProperties: r { score: coalesce(r.score, 0.0) }
    },
    { undirectedRelationshipTypes: ['HAS_DISEASE']}                                    
    )""")
    
assert G_control.node_count() == result_test["nodeCount"]

In [18]:
#gds.graph.drop("control_graph")

gds.graph.list()

,degreeDistribution,graphName,database,databaseLocation,memoryUsage,sizeInBytes,nodeCount,relationshipCount,configuration,density,creationTime,modificationTime,schema,schemaWithOrientation
0,"{'min': 0, 'max': 1970, 'p90': 0, 'p999': 504,...",control_graph,neo4j,local,252 MiB,264855176,255581,1960445,"{'readConcurrency': 4, 'undirectedRelationship...",0.000030,2024-11-14T07:40:48.055576000+00:00,2024-11-14T07:41:17.558623000+00:00,"{'graphProperties': {}, 'nodes': {'Phenotype':...","{'graphProperties': {}, 'nodes': {'Phenotype':..."
1,"{'min': 0, 'max': 1970, 'p90': 1, 'p999': 504,...",test_graph,neo4j,local,237 MiB,248733928,255581,2013958,"{'readConcurrency': 4, 'undirectedRelationship...",0.000031,2024-11-14T07:40:11.462074000+00:00,2024-11-14T07:52:50.567157000+00:00,"{'graphProperties': {}, 'nodes': {'Phenotype':...","{'graphProperties': {}, 'nodes': {'Phenotype':..."
2,"{'min': 0, 'max': 1970, 'p90': 2, 'p999': 504,...",train_graph,neo4j,local,314 MiB,329381472,255795,2043896,"{'readConcurrency': 4, 'undirectedRelationship...",0.000031,2024-11-14T07:39:23.617595000+00:00,2024-11-14T07:49:54.178445000+00:00,"{'graphProperties': {}, 'nodes': {'Phenotype':...","{'graphProperties': {}, 'nodes': {'Phenotype':..."


##### approximate inductive link prediction using FastRP for node embedding (Cypher)

In [8]:
# for inductive link prediction, FastRP requires propertyRatio = 1.0 and a random seed
## positive value of propertyRatio requires featureProperties to be non-empty
### run Louvain (community detection/ hierachical clustering algorithm) [-> consider other algorithms later]

In [9]:
## did not yield meaningful predictions -> skip

#  Louvain algorithm for train_graph

#gds.run_cypher("""CALL gds.louvain.mutate("train_graph", {maxIterations: 10, mutateProperty: "community"}) YIELD nodePropertiesWritten""")

#  Louvain algorithm for test_graph

#gds.run_cypher("""CALL gds.louvain.mutate("test_graph", {maxIterations: 10, mutateProperty: "community"}) YIELD nodePropertiesWritten""")




In [10]:
## only works for undirected graphs -> skip Leiden for now

#  Leiden algorithm for train_graph
#gds.run_cypher("""CALL gds.leiden.mutate("train_graph", {mutateProperty: "leiden", relationshipWeightProperty: 'score', randomSeed: 25}) YIELD communityCount""")

#  Leiden algorithm for test_graph
#gds.run_cypher("""CALL gds.louvain.mutate("test_graph", {mutateProperty: "leiden", relationshipWeightProperty: 'score', randomSeed: 25}) YIELD communityCount""")


In [ ]:
# similarity algorithms 
# -> in stream mode, computes the similarity between pairs of nodes in the graph (here)
# -> in mutate mode, generates new relationships between pairs of nodes in the projected graph and assigns a similarity score to each relationship (next cell)
# Node Similarity train_graph
sim = gds.run_cypher("""CALL gds.nodeSimilarity.stream('train_graph', 
               {
               //nodeLabels: ['Biological_sample'], 
               relationshipWeightProperty: 'score' 
               //topK: 1
               }) 
               YIELD node1, node2, similarity 
               //WHERE labels(gds.util.asNode(node1)) = ['Biological_sample'] AND labels(gds.util.asNode(node2)) = ['Biological_sample']
               RETURN labels(gds.util.asNode(node1)) AS node1, labels(gds.util.asNode(node2)) AS node2, similarity""")

In [12]:
sim

,node1,node2,similarity
0,[Protein],[Protein],0.483899
1,[Protein],[Protein],0.337599
2,[Protein],[Protein],0.336660
3,[Protein],[Protein],0.255748
4,[Protein],[Protein],0.242625
...,...,...,...
189607,[Biological_sample],[Protein],0.001266
189608,[Biological_sample],[Protein],0.001259
189609,[Biological_sample],[Protein],0.001243
189610,[Biological_sample],[Protein],0.001230


In [13]:
# similarity algorithms

# Node Similarity train_graph
gds.run_cypher("""CALL gds.nodeSimilarity.mutate('train_graph', 
               {
               mutateRelationshipType: 'SIMILAR_TO',
               mutateProperty: 'score', 
               //relationshipWeightProperty: 'score', 
               topK: 1
               }) 
               YIELD nodesCompared, relationshipsWritten
              """)

,nodesCompared,relationshipsWritten
0,45662,44413


In [14]:
# similarity algorithms

# Node Similarity test_graph
gds.run_cypher("""CALL gds.nodeSimilarity.mutate('test_graph', 
               {
               mutateRelationshipType: 'SIMILAR_TO',
               mutateProperty: 'score',  
               //relationshipWeightProperty: 'score', 
               topK: 1}) 
               YIELD nodesCompared, relationshipsWritten""")

,nodesCompared,relationshipsWritten
0,34836,34024


In [ ]:
## try Label Propagation algorithm for community detection

#gds.run_cypher = ("""CALL gds.graph.nodeProperties.drop('train_graph', ['LabelProp'])
#YIELD propertiesRemoved""")


In [21]:

#  Label Propagation algorithm for train_graph
gds.run_cypher("""CALL gds.labelPropagation.mutate('train_graph', 
               { 
               mutateProperty: 'LabelProp', 
               relationshipWeightProperty: 'score', 
               relationshipTypes: ['SIMILAR_TO']
               //relationshipTypes: ['HAS_PHENOTYPE','HAS_DAMAGE','HAS_PROTEIN','COMPILED_INTERACTS_WITH','HAS_DISEASE','IS_BIOMARKER_OF_DISEASE','HAS_PARENT'] 
               }) 
               YIELD communityCount, ranIterations, didConverge""")


,communityCount,ranIterations,didConverge
0,219991,8,True


In [22]:
## caution relationshipTypes: algorithm won't work if a relationshipType is not present in the graph

#  Label Propagation algorithm for test_graph
gds.run_cypher("""CALL gds.labelPropagation.mutate('test_graph', 
               { mutateProperty: 'LabelProp', 
               relationshipWeightProperty: 'score', 
               relationshipTypes: ['SIMILAR_TO']
               //relationshipTypes: ['HAS_PHENOTYPE','HAS_DAMAGE','HAS_PROTEIN','COMPILED_INTERACTS_WITH','HAS_DISEASE','HAS_PARENT'] 
               }) 
               YIELD communityCount, ranIterations, didConverge""")

,communityCount,ranIterations,didConverge
0,228372,8,True


In [23]:
# degree centrality

#  Degree centrality for train_graph
gds.run_cypher("""CALL gds.degree.mutate('train_graph', { mutateProperty: 'degree', relationshipWeightProperty: 'score' }) YIELD centralityDistribution, nodePropertiesWritten""")

#  Degree centrality for test_graph
gds.run_cypher("""CALL gds.degree.mutate('test_graph', { mutateProperty: 'degree', relationshipWeightProperty: 'score' }) YIELD centralityDistribution, nodePropertiesWritten""")

,centralityDistribution,nodePropertiesWritten
0,"{'min': 0.0, 'max': 1355.4687499403954, 'p90':...",255581


In [24]:
# Node2Vec node embeddings for train_graph

gds.run_cypher("""CALL gds.node2vec.mutate("train_graph",
    {mutateProperty: 'node2vec',
    walkLength: 10,
    walksPerNode: 10,
    relationshipWeightProperty: 'score',
    //relationshipTypes: ['HAS_SIMILARITY'],           
    iterations: 10,
    randomSeed: 42 }) YIELD nodePropertiesWritten
    """)


,nodePropertiesWritten
0,255795


In [25]:

# Node2Vec node embeddings for test_graph

gds.run_cypher("""CALL gds.node2vec.mutate("test_graph",
    {mutateProperty: 'node2vec',
    walkLength: 10,
    walksPerNode: 10,
    relationshipWeightProperty: 'score',
    //relationshipTypes: ['HAS_SIMILARITY'],           
    iterations: 10,
    randomSeed: 42}) YIELD nodePropertiesWritten
    """)

,nodePropertiesWritten
0,255581


In [ ]:
# failed approach to split the links-to-predict into a holdout set and a remaining set, i. e. train and test set
# -> did not work as expected
# -> not pursued further

gds.run_cypher("""CALL gds.alpha.ml.splitRelationships.mutate('train_graph',
               {relationshipTypes: ['HAS_DISEASE'],
               holdoutFraction: 0.2,
               negativeSamplingRatio: 100.0,
               holdoutRelationshipType: 'DISEASE_HOLDOUT',
               remainingRelationshipType: 'DISEASE_REMAINING',
               randomSeed: 6667,
               relationshipWeightProperty: 'score'
               }) 
               YIELD relationshipsWritten""")

,relationshipsWritten
0,2069


In [ ]:
# pipeline configuration - Cypher

if gds.run_cypher("""CALL gds.pipeline.exists('pipe_fastrp') YIELD exists""").iloc[0,0]==True:
    gds.run_cypher("""CALL gds.pipeline.drop('pipe_fastrp')""")

#create pipeline
gds.beta.pipeline.linkPrediction.create('pipe_fastrp')

# add node property
# for inductive link prediction with FastRP node embeddings, propertyRatio = 1.0 and a random seed are required
# -> positive value of propertyRatio requires featureProperties to be non-empty (here: 'LabelProp')
gds.run_cypher(""" CALL gds.beta.pipeline.linkPrediction.addNodeProperty('pipe_fastrp', 'fastRP', {
    mutateProperty: 'embedding',
    embeddingDimension: 256,
    randomSeed: 42,
    propertyRatio: 1.0,
    featureProperties: ['LabelProp'],
    relationshipWeightProperty: 'score',
    contextNodeLabels: ['Protein', 'Gene', 'Phenotype'],
    contextRelationshipTypes: ['HAS_PROTEIN', 'HAS_DAMAGE', 'COMPILED_INTERACTS_WITH', 'HAS_PARENT', 'HAS_PHENOTYPE', 'IS_BIOMARKER_OF_DISEASE', 'SIMILAR_TO']
})""")


# add link features - also add 'community' and 'node2vec' as nodeProperties?
gds.run_cypher(""" CALL gds.beta.pipeline.linkPrediction.addFeature('pipe_fastrp', 'cosine', {
    nodeProperties: ['embedding']
})""")

#Configuring the relationship split 
gds.run_cypher(""" CALL gds.beta.pipeline.linkPrediction.configureSplit('pipe_fastrp', {
    testFraction: 0.3,
    trainFraction: 0.7,
    validationFolds: 3,
    negativeSamplingRatio: 100.0            
})""")


# add model candidates
gds.run_cypher(""" CALL gds.beta.pipeline.linkPrediction.addLogisticRegression('pipe_fastrp')""")
gds.run_cypher(""" CALL gds.beta.pipeline.linkPrediction.addRandomForest('pipe_fastrp', {numberOfDecisionTrees: 100})""")
gds.run_cypher(""" CALL gds.alpha.pipeline.linkPrediction.addMLP('pipe_fastrp', {hiddenLayerSizes: [64, 32], penalty: 0.01, patience: 2})""")

# memory estimation
#gds.run_cypher(""" CALL gds.beta.pipeline.linkPrediction.train.estimate('train_graph', {
#               pipeline: 'pipe_fastrp',
#               modelName: 'pheno-fastrp',
#               targetRelationshipType: 'HAS_PHENOTYPE'
#               })""")

if gds.run_cypher("""CALL gds.model.exists('pheno-fastrp') YIELD exists""").iloc[0,0]==True:
    gds.run_cypher("""CALL gds.model.drop('pheno-fastrp')""")

# training
gds.run_cypher("""CALL gds.beta.pipeline.linkPrediction.train('train_graph', {
  pipeline: 'pipe_fastrp',
  modelName: 'pheno-fastrp',
  metrics: ['AUCPR', 'OUT_OF_BAG_ERROR'],
  negativeClassWeight: 0.01,             
  sourceNodelabel: 'Biological_sample',
  targetNodeLabel: 'Disease',             
  targetRelationshipType: 'HAS_DISEASE',
  randomSeed: 42
}) YIELD modelInfo, modelSelectionStats
RETURN
  modelInfo.bestParameters AS winningModel,
  modelInfo.metrics.AUCPR.train.avg AS avgTrainScore,
  modelInfo.metrics.AUCPR.outerTrain AS outerTrainScore,
  modelInfo.metrics.AUCPR.test AS testScore,
  [cand IN modelSelectionStats.modelCandidates | cand.metrics.AUCPR.validation.avg] AS validationScores""")



,winningModel,avgTrainScore,outerTrainScore,testScore,validationScores
0,"{'maxDepth': 2147483647, 'criterion': 'GINI', ...",0.502277,0.554412,0.473178,"[0.49267846618151406, 0.5484881268739925, 0.49..."


In [ ]:
model = gds.run_cypher("""CALL gds.model.list()""")

model


,modelName,modelType,modelInfo,creationTime,trainConfig,graphSchema,loaded,stored,published
0,pheno-hashgnn,LinkPrediction,{'metrics': {'AUCPR': {'test': 0.7552452688702...,2024-11-13T10:59:47.422371000+00:00,"{'randomSeed': 42, 'targetRelationshipType': '...","{'graphProperties': {}, 'nodes': {'Disease': {...",True,False,False
1,pheno-fastrp,LinkPrediction,{'metrics': {'OUT_OF_BAG_ERROR': {'test': 0.52...,2024-11-13T15:20:05.851813000+00:00,"{'randomSeed': 42, 'targetRelationshipType': '...","{'graphProperties': {}, 'nodes': {'Disease': {...",True,False,False
2,graphsage,graphSage,"{'metrics': {'didConverge': False, 'ranIterati...",2024-11-13T10:50:59.774297000+00:00,"{'aggregator': 'MEAN', 'jobId': '799b7747-3e32...","{'graphProperties': {}, 'nodes': {'Phenotype':...",True,False,False
3,pheno-sage,LinkPrediction,{'metrics': {'OUT_OF_BAG_ERROR': {'test': 0.37...,2024-11-13T10:51:10.420449000+00:00,"{'randomSeed': 42, 'targetRelationshipType': '...","{'graphProperties': {}, 'nodes': {'Disease': {...",True,False,False


In [54]:
## make predictions on test_graph

predict_fastrp =  gds.run_cypher("""CALL gds.beta.pipeline.linkPrediction.predict.stream('test_graph', {
  modelName: 'pheno-fastrp',
  topN: 100         
  //threshold: 0.1
  //topK: 1,                               
  //sampleRate: 0.9
})
 YIELD node1, node2, probability
 //WHERE gds.util.asNode(node1).id STARTS WITH "HP:0000"
 //WHERE gds.util.asNode(node2).subjectid STARTS WITH "42066"
 //RETURN DISTINCT gds.util.asNode(node2).subjectid AS sample, COLLECT(DISTINCT gds.util.asNode(node1).id) AS disease, COUNT(DISTINCT gds.util.asNode(node1).id) AS count
 RETURN gds.util.asNode(node1).id AS disease_id, 
 gds.util.asNode(node2).subjectid AS patient_id
 ORDER BY gds.util.asNode(node2).subjectid""")

In [55]:
predict_fastrp

,disease_id,patient_id
0,DOID:0060694,42033
1,DOID:0110462,42033
2,DOID:14449,42033
3,DOID:6511,42033
4,DOID:6511,42066
...,...,...
95,DOID:6511,44154
96,DOID:6511,44212
97,DOID:14449,44212
98,DOID:6511,44222


In [49]:
ctrl = gds.run_cypher("""MATCH (bs:Biological_sample)
               WHERE bs.subjectid STARTS WITH "42" OR bs.subjectid STARTS WITH "43" OR bs.subjectid STARTS WITH "44"
               MATCH (bs)-[:HAS_DISEASE]->(d:Disease)
               RETURN d.id as disease_id, bs.subjectid as patient_id
               ORDER BY patient_id""")
ctrl

,disease_id,patient_id
0,DOID:10030,42075
1,DOID:6376,42075
2,DOID:10030,42135
3,DOID:4372,42199
4,DOID:112,42281
5,DOID:1680,42281
6,DOID:10030,42281
7,DOID:5295,42281
8,DOID:10230,42292
9,DOID:11963,42292


In [58]:
import pandas as pd

# Step 1: Merge DataFrames on both disease_id and patient_id
correct_predictions = pd.merge(predict_fastrp, ctrl, on=['disease_id', 'patient_id'])

# Step 2: Identify false positives (predictions not in actual data)
false_positives = pd.merge(predict_fastrp, correct_predictions, how='left', indicator=True)
false_positives = false_positives[false_positives['_merge'] == 'left_only'].drop(columns=['_merge'])

# Step 3: Identify false negatives (actual data not in predictions)
false_negatives = pd.merge(ctrl, correct_predictions, how='left', indicator=True)
false_negatives = false_negatives[false_negatives['_merge'] == 'left_only'].drop(columns=['_merge'])

# Display results
print("Correct Predictions:")
print(correct_predictions)

print("\nFalse Positives (Predicted but not actual):")
print(false_positives)

print("\nFalse Negatives (Actual but not predicted):")
print(false_negatives)

Correct Predictions:
Empty DataFrame
Columns: [disease_id, patient_id]
Index: []

False Positives (Predicted but not actual):
      disease_id patient_id
0   DOID:0060694      42033
1   DOID:0110462      42033
2     DOID:14449      42033
3      DOID:6511      42033
4      DOID:6511      42066
..           ...        ...
95     DOID:6511      44154
96     DOID:6511      44212
97    DOID:14449      44212
98     DOID:6511      44222
99     DOID:6511      44228

[100 rows x 2 columns]

False Negatives (Actual but not predicted):
      disease_id patient_id
0     DOID:10030      42075
1      DOID:6376      42075
2     DOID:10030      42135
3      DOID:4372      42199
4       DOID:112      42281
5      DOID:1680      42281
6     DOID:10030      42281
7      DOID:5295      42281
8     DOID:10230      42292
9     DOID:11963      42292
10    DOID:11963      42302
11    DOID:11476      42302
12     DOID:9651      42302
13  DOID:0050778      42321
14    DOID:10914      42321
15     DOID:2089     

In [57]:
# Step 1: Group by patient_id and collect sets of disease_id
predicted_groups = predict_fastrp.groupby('patient_id')['disease_id'].apply(set).reset_index()
actual_groups = ctrl.groupby('patient_id')['disease_id'].apply(set).reset_index()

# Step 2: Merge on patient_id to align predicted and actual disease sets per patient
merged_df = pd.merge(predicted_groups, actual_groups, on='patient_id', how='outer', suffixes=('_predicted', '_actual'))

# Step 3: Fill NaN values with empty sets
merged_df['disease_id_predicted'] = merged_df['disease_id_predicted'].apply(lambda x: x if isinstance(x, set) else set())
merged_df['disease_id_actual'] = merged_df['disease_id_actual'].apply(lambda x: x if isinstance(x, set) else set())

# Step 4: Check for any overlap in disease sets for each patient
merged_df['has_overlap'] = merged_df.apply(lambda row: bool(row['disease_id_predicted'] & row['disease_id_actual']), axis=1)

# Display results
print("Disease Prediction Comparison:")
print(merged_df[['patient_id', 'disease_id_predicted', 'disease_id_actual', 'has_overlap']])

Disease Prediction Comparison:
   patient_id                               disease_id_predicted  \
0       42033  {DOID:14449, DOID:0060694, DOID:0110462, DOID:...   
1       42066  {DOID:14449, DOID:0060694, DOID:0110462, DOID:...   
2       42075  {DOID:14449, DOID:0060694, DOID:0110462, DOID:...   
3       42135  {DOID:14449, DOID:0060694, DOID:0110462, DOID:...   
4       42199  {DOID:14449, DOID:0060694, DOID:0110462, DOID:...   
5       42231  {DOID:14449, DOID:0060694, DOID:0110462, DOID:...   
6       42275  {DOID:14449, DOID:0060694, DOID:0110462, DOID:...   
7       42281  {DOID:14449, DOID:0060694, DOID:0110462, DOID:...   
8       42292  {DOID:14449, DOID:0060694, DOID:0110462, DOID:...   
9       42302  {DOID:14449, DOID:0060694, DOID:0110462, DOID:...   
10      42321  {DOID:14449, DOID:0060694, DOID:0110462, DOID:...   
11      42346  {DOID:14449, DOID:0060694, DOID:0110462, DOID:...   
12      42367  {DOID:14449, DOID:0060694, DOID:0110462, DOID:...   
13      42412  {D

In [56]:
# Step 1: Extract unique disease IDs as sets
predicted_diseases = set(predict_fastrp['disease_id'].unique())
actual_diseases = set(ctrl['disease_id'].unique())

# Step 2: Check for overlap
overlap = predicted_diseases & actual_diseases  # Intersection of both sets

# Results
if overlap:
    print("Overlap exists. The following disease IDs are present in both predicted and actual data:")
    print(overlap)
else:
    print("No overlap found between predicted and actual disease IDs.")

No overlap found between predicted and actual disease IDs.


In [56]:
## set up pipeline using PythonClient for comparison

if gds.run_cypher("""CALL gds.pipeline.exists('pipe_fastrp_cli') YIELD exists""").iloc[0,0]==True:
    gds.run_cypher("""CALL gds.pipeline.drop('pipe_fastrp_cli')""")

#create pipeline
pipe_fastrp_cli = gds.lp_pipe("pipe_fastrp_cli")

#Add FastRP as a property step producing "embedding" node properties
#pipe.addNodeProperty("fastRP", embeddingDimension=256, mutateProperty="embedding", randomSeed=42, contextNodelLabels=["Protein", "Gene"], contextRelationshipTypes=["HAS_PROTEIN", "HAS_DAMAGE"])

gds.run_cypher("""CALL gds.beta.pipeline.linkPrediction.addNodeProperty('pipe_fastrp_cli', 'fastRP', {
    mutateProperty: 'embedding',
    embeddingDimension: 256,
    randomSeed: 42,
    propertyRatio: 1.0,
    featureProperties: ['community'],
    contextNodeLabels: ['Protein', 'Gene'],
    contextRelationshipTypes: ['HAS_PROTEIN', 'HAS_DAMAGE']
    })""")

# Combine our "embedding" node properties with Hadamard to create link features for training
pipe_fastrp_cli.addFeature("hadamard", nodeProperties=["embedding"])

# Verify that the features to be used in model training are what we expect
steps = pipe_fastrp_cli.feature_steps()


# Specify the fractions we want for our dataset split
pipe_fastrp_cli.configureSplit(trainFraction=0.6, testFraction=0.2, validationFolds=3)

# Add a random forest model with tuning over `maxDepth`
pipe_fastrp_cli.addRandomForest(numberOfDecisionTrees=100)
pipe_fastrp_cli.addMLP(hiddenLayerSizes=[64, 32], penalty=0.01, patience=2)
pipe_fastrp_cli.addLogisticRegression()


name                                                   pipe_fastrp_cli
nodePropertySteps    [{'name': 'gds.fastRP.mutate', 'config': {'ran...
featureSteps         [{'name': 'HADAMARD', 'config': {'nodeProperti...
splitConfig          {'testFraction': 0.2, 'validationFolds': 3, 't...
autoTuningConfig                                     {'maxTrials': 10}
parameterSpace       {'MultilayerPerceptron': [{'minEpochs': 1, 'ma...
Name: 0, dtype: object

In [57]:
if gds.run_cypher("""CALL gds.model.exists("pheno_fastrp_cli") YIELD exists""").iloc[0,0]==True:
    gds.run_cypher("""CALL gds.model.drop("pheno_fastrp_cli")""")

In [58]:
# Train a model named "pheno-fastrp-cli"

pheno_fastrp_cli, train_result = pipe_fastrp_cli.train(
        G_train,
        modelName="pheno_fastrp_cli",
        metrics = ["AUCPR", "OUT_OF_BAG_ERROR"],
        sourceNodeLabel="Biological_sample",
        targetNodeLabel="Disease",
        targetRelationshipType="HAS_DISEASE",
        randomSeed=42
    )

ClientError: {code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.beta.pipeline.linkPrediction.train`: Caused by: java.lang.IllegalStateException: Storing more than `3` models in the catalog is available with a licensed Graph Data Science library. See documentation at https://neo4j.com/docs/graph-data-science/}

In [ ]:
# make predictions

predict_fastrp_cli = pheno_fastrp_cli.predict_stream(G_test, topN=500, sampleRate=1.0, threshold=0.1)

predict_fastrp_cli

In [ ]:
metric = gds.run_cypher( """ CALL gds.model.list()
YIELD modelName, modelType, modelInfo, trainConfig, graphSchema
//WHERE modelName = "pheno"
RETURN  modelInfo.bestParameters AS winningModel, modelName, modelType, modelInfo, trainConfig, graphSchema""")

metric